# 第68章 交互散点图（px.scatter）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 5 / 18 步：交互探索趋势、类别和变量关系**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互柱状图（px.bar）  →  **本章任务：** 交互散点图（px.scatter）  →  **下一步：** 交互气泡图
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

在很多数据分析场景里，手上常常只有两列数值——比如购买件数和客单价——想知道它们之间有没有关联、哪些点又明显偏离整体。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交互散点图（px.scatter）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互散点图（px.scatter）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互散点图（px.scatter）」并读出其中的结论。


## 适用场景

**背景引入**：在很多数据分析场景里，手上常常只有两列数值——比如购买件数和客单价——想知道它们之间有没有关联、哪些点又明显偏离整体。静态的散点图只能看一眼大致分布，一旦点数变多、想单独把某几个特殊点看清楚就很吃力。Plotly 的交互散点图把缩放、悬浮取数和点选都做成内置能力，可以随时放大一块区域、划到任一点上读出它的具体数值，边看图边排查异常点就顺手很多。（可以把它想成一张点名表：表格里每一行是一个人，x 是其中一项成绩、y 是另一项，散点图就把这一行画成一个点；图上几个点，表格就有几行，放大、悬停都是为了看清某个“人”到底是谁、哪里不对。）

分析两个数值变量关系，需要交互识别异常点。


## 数据结构

两列数值和可选分类列；每行一条观察。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 opacity=0.7 改为 0.4 或 1.0，观察透明度对重叠点显示的影响
2. 添加 symbol="channel" 参数，对比颜色映射与形状映射的组合区分效果
3. 修改 hover_data 增加或移除字段，说明悬浮信息内容对交互检查的作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.scatter()`、`fig.update_layout()`、`fig.show()` | 分析两个数值变量关系，需要交互识别异常点。 | 点太多导致前端卡顿 |
| 进阶变体 | `px.scatter()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | Hover字段泄露无关或敏感信息 |
| 关键参数 | `color` | 分类 | 点太多导致前端卡顿 |
| 关键参数 | `symbol` | 形状 | Hover字段泄露无关或敏感信息 |
| 关键参数 | `opacity` | 透明度 | 只看局部缩放后忘记恢复全局 |
| 关键参数 | `hover_data` | 悬浮字段 | 点太多导致前端卡顿 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-68 -->
### 数学推导｜散点关系与 Pearson 相关系数

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先去掉量纲。** 标准化后 $z_{xi}=(x_i-\bar{x})/s_x$、$z_{yi}=(y_i-\bar{y})/s_y$。

**第 2 步｜看同一观测上的方向是否一致。** 当两个标准化值同号时，乘积 $z_{xi}z_{yi}$ 为正；异号时为负。

**第 3 步｜对共同变化求平均。** 样本相关可写为

$$
r=\frac{1}{n-1}\sum_{i=1}^{n}z_{xi}z_{yi}
$$

展开标准化定义，就得到分子为离差乘积、分母为两个平方和平方根的常见形式。

**把上面的关系收束为本章计算式：**

$$
r=\frac{\sum_i(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_i(x_i-\bar{x})^2\sum_i(y_i-\bar{y})^2}}
$$

**符号解释：** $r\in[-1,1]$ 描述线性共同变化的方向和强度。

**代码对应：** `df[[x, y]].corr().iloc[0, 1]` 与散点图配合使用。

**使用边界：** 相关不等于因果；异常值、非线性和分组结构都可能改变总体相关。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.scatter(
    orders,
    x="items",
    y="order_value",
    color="category",
    hover_data=["region", "channel"],
    opacity=0.7,
    title="购买件数与客单价",
)
fig.update_layout(
    xaxis_title="购买件数", yaxis_title="客单价（元）", legend_title="品类"
)
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：复制上面「61.4 基础图表」的散点图，把分组方式从颜色换成形状。把 “color="category"” 改成 “symbol="channel"”（先删掉 color 参数），保留 title 与 opacity，运行后观察并记录：同一张图改用形状编码后，类别之间的区分是否更清楚？颜色映射和形状映射各适合什么情况？


In [ ]:
try:
    # 请在下方填写代码
    import plotly.express as px

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.scatter(
    orders,
    x="order_value",
    y="sales",
    color="region",
    symbol="channel",
    hover_name="category",
    opacity=0.65,
    title="客单价与订单销售额",
)
fig.update_layout(
    xaxis_title="客单价（元）",
    yaxis_title="订单销售额（元）",
    legend_title="区域 / 渠道",
)
fig.show()


## 参数说明

- color：分类
- symbol：形状
- opacity：透明度
- hover_data：悬浮字段


## 结果解读

通过缩放查看密集区域，Hover确认异常观察的类别和精确值。


## 本章实训：交互图与信息层次

这一组实验专门训练"观察一个结果 → 只改一个变量 → 解释变化"。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 点太多导致前端卡顿
- Hover字段泄露无关或敏感信息
- 只看局部缩放后忘记恢复全局


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：用 size 编码销售额，把散点升级为气泡图
    # 【目标】用点的大小再编码一列，让散点承载三个维度。
    import plotly.express as px

    # 起点示例(已可运行)：加 size="sales"，点的大小编码总销售额。
    fig = px.scatter(
        orders,
        x="items",
        y="order_value",
        color="category",
        size="sales",
        size_max=25,
        opacity=0.7,
        title="购买件数、客单价与销售额",
    )
    fig.update_layout(
        xaxis_title="购买件数", yaxis_title="客单价（元）", legend_title="品类"
    )
    fig.show()

    # ---- 反思记录：加上大小编码，读图时多了哪层信息 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用交互散点图查看二维关系并通过Hover检查单个观察。


### 你已经掌握

- 判断交互散点图（px.scatter）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `color` | 分类 |
| `symbol` | 形状 |
| `opacity` | 透明度 |
| `hover_data` | 悬浮字段 |


### 需要注意

- 点太多导致前端卡顿
- Hover字段泄露无关或敏感信息
- 只看局部缩放后忘记恢复全局


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
import plotly.express as px

fig = px.scatter(
    orders,
    x="items",
    y="order_value",
    symbol="channel",
    hover_data=["region", "channel"],
    opacity=0.7,
    title="购买件数与客单价",
)
fig.update_layout(
    xaxis_title="购买件数", yaxis_title="客单价（元）", legend_title="渠道"
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
sample = orders.nlargest(80, "sales")
fig = px.scatter(
    sample,
    x="items",
    y="sales",
    color="category",
    hover_data={"order_value": ":.1f", "region": True},
    title="高销售订单关系",
)
fig.update_layout(xaxis_title="购买件数", yaxis_title="订单销售额（元）")
fig.show()
